# Attention-LSTM Training
This notebook trains a score-prediction model from over-by-over engineered data.

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Masking, LSTM, Dense, Attention, GlobalAveragePooling1D
from tensorflow.keras.models import Model

PROCESSED_FILE = 'processed_data.csv'
MODEL_FILE = 'ball_outcome_model.keras'

print('TensorFlow version:', tf.__version__)

I0000 00:00:1774642052.157218   15858 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774642052.582842   15858 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774642053.839121   15858 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow version: 2.21.0


In [2]:
def create_3d_sequences(processed_df, max_overs=20, num_features=6):
    import numpy as np

    print("Creating expanding window sequences to prevent data leakage...")

    feature_cols = [
        "runs_this_over",
        "wickets_this_over",
        "cumulative_runs",
        "cumulative_wickets",
        "run_rate",
        "balls_remaining",
    ]

    # Optional safety: keep chronological order if an over column exists
    if "over" in processed_df.columns:
        processed_df = processed_df.sort_values(["match_id", "over"], kind="stable")

    grouped = list(processed_df.groupby("match_id", sort=False))
    total_samples = sum(max(len(g) - 1, 0) for _, g in grouped)

    X = np.zeros((total_samples, max_overs, num_features), dtype=np.float32)
    y = np.empty((total_samples, 1), dtype=np.float32)

    pos = 0
    for _, match_data in grouped:
        features = match_data[feature_cols].to_numpy(dtype=np.float32, copy=False)
        overs_played = features.shape[0]
        if overs_played <= 1:
            continue

        if overs_played > max_overs + 1:
            # Keep behavior safe for longer innings
            features = features[: max_overs + 1]
            overs_played = features.shape[0]

        final_score = float(match_data["final_score"].iloc[0])
        n = overs_played - 1  # number of snapshots

        # Build all expanding snapshots for this match
        match_X = np.zeros((n, max_overs, num_features), dtype=np.float32)

        # Vectorized fill:
        # row j is present in snapshots j..n-1 at timestep j
        for j in range(n):
            match_X[j:, j, :] = features[j]

        X[pos : pos + n] = match_X
        y[pos : pos + n, 0] = final_score
        pos += n

    return X, y

def build_model(time_steps=20, num_features=6):
    inputs = Input(shape=(time_steps, num_features))

    # Ignore 0.0 padded values
    masked_inputs = Masking(mask_value=0.0)(inputs)

    # LSTM output for every timestep
    lstm_out = LSTM(64, return_sequences=True)(masked_inputs)

    # Self-attention
    attention_out = Attention()([lstm_out, lstm_out])

    # Sequence compression
    pooled_out = GlobalAveragePooling1D()(attention_out)

    # Regression head
    x = Dense(32, activation='relu')(pooled_out)
    outputs = Dense(1, activation='linear')(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='mae', metrics=['mae'])
    return model

In [3]:
processed_df = pd.read_csv(PROCESSED_FILE)
print('Processed shape:', processed_df.shape)

X, y = create_3d_sequences(processed_df)

# Train/validation split (80/20)
split_idx = int(0.8 * len(X))
X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

print('Training data shape:', X_train.shape)
print('Validation data shape:', X_val.shape)

Processed shape: (95025, 9)
Creating expanding window sequences to prevent data leakage...
Training data shape: (72089, 20, 6)
Validation data shape: (18023, 20, 6)


In [4]:
model = build_model()
model.summary()

W0000 00:00:1774642069.256678   15858 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 20, 6)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 20, 6)     │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masking (Masking)   │ (None, 20, 6)     │          0 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any (Any)           │ (None, 20)        │          0 │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 20, 64)    │     18,176 │ masking[0][0],    │
│                     │                   │            │ any[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 20, 64)    │          0 │ lstm[0][0],       │
│ (Attention)         │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convert_to_tensor   │ (None, 20)        │          0 │ any[0][0]         │
│ (ConvertToTensor)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ attention[0][0],  │
│ (GlobalAveragePool… │                   │            │ convert_to_tenso… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │      2,080 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │         33 │ dense[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 20,289 (79.25 KB)

 Trainable params: 20,289 (79.25 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
print('Starting Training (Targeting MAE < 10)...')
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    verbose=1
)

Starting Training (Targeting MAE < 10)...
Epoch 1/30
  16/2253 ━━━━━━━━━━━━━━━━━━━━ 15s 7ms/step - loss: 141.7738 - mae: 141.7738

E0000 00:00:1774642075.524738   15858 util.cc:131] oneDNN supports DT_BOOL only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


2253/2253 ━━━━━━━━━━━━━━━━━━━━ 22s 9ms/step - loss: 28.5511 - mae: 28.5511 - val_loss: 19.3275 - val_mae: 19.3275
Epoch 2/30
2253/2253 ━━━━━━━━━━━━━━━━━━━━ 21s 9ms/step - loss: 18.3508 - mae: 18.3508 - val_loss: 18.1870 - val_mae: 18.1870
Epoch 3/30
2253/2253 ━━━━━━━━━━━━━━━━━━━━ 21s 10ms/step - loss: 18.1892 - mae: 18.1892 - val_loss: 18.2509 - val_mae: 18.2509
Epoch 4/30
2253/2253 ━━━━━━━━━━━━━━━━━━━━ 23s 10ms/step - loss: 18.1147 - mae: 18.1147 - val_loss: 17.9672 - val_mae: 17.9672
Epoch 5/30
2253/2253 ━━━━━━━━━━━━━━━━━━━━ 22s 10ms/step - loss: 18.0750 - mae: 18.0750 - val_loss: 18.0428 - val_mae: 18.0428
Epoch 6/30
2253/2253 ━━━━━━━━━━━━━━━━━━━━ 22s 10ms/step - loss: 18.0135 - mae: 18.0135 - val_loss: 17.8364 - val_mae: 17.8364
Epoch 7/30
2253/2253 ━━━━━━━━━━━━━━━━━━━━ 22s 10ms/step - loss: 17.9873 - mae: 17.9873 - val_loss: 18.3879 - val_mae: 18.3879
Epoch 8/30
2253/2253 ━━━━━━━━━━━━━━━━━━━━ 22s 10ms/step - loss: 17.9561 - mae: 17.9561 - val_loss: 17.9882 - val_mae: 17.9882
Epoch

In [6]:
val_loss, val_mae = model.evaluate(X_val, y_val, verbose=0)
print(f'Validation MAE: {val_mae:.4f}')

model.save(MODEL_FILE)
print(f'Saved trained model to {MODEL_FILE}')

Validation MAE: 18.0750
Saved trained model to ball_outcome_model.keras
